# Conexión a InfluxDB con variables de entorno y `.env`

> _Tutorial · Caso de uso: **Overview** · Capa Medallion: **transversal** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Disponer de una plantilla reutilizable para conectar con InfluxDB leyendo `.env`. Sin secretos en código. Un fallback claro para clase si el stack no está levantado.


## 2. Qué se aprende

- Cómo cargar `.env` con `python-dotenv` (o nuestro fallback).
- Patrón `client = get_influx_client()` con el helper del repo.
- Cómo distinguir modo offline (mock) y online (real).
- Smoke query Flux mínima de salud.
- Por qué nunca commitear el `.env`.


## 3. Contexto del caso de uso

Cualquier notebook posterior asume que esta conexión funciona. Si estás en clase sin Influx levantado, declara `INFLUX_OFFLINE=true` y los notebooks `needs-stack` mostrarán datos mockeados.


## 4. Relación con CENTINELA+

El cliente que usaremos es **idéntico** al que usará CENTINELA+ contra `simarro-prod`. La única diferencia entre desarrollo y producción es la URL y el token. Esto es el corazón de la regla *cambio de credenciales, no reescritura*.


## 5. Relación con Medallion

Este notebook es la **puerta de entrada a la capa plata**. Sin esta conexión nadie puede leer ni escribir.


## 6. Datos de entrada

El `.env` del repo (`.env.example` como referencia). Variables esperadas: `INFLUXDB_URL`, `INFLUXDB_TOKEN`, `INFLUXDB_ORG`, `INFLUXDB_BUCKET`.


## 7. Schema CAPTIA esperado

El cliente solo se conecta al servidor; no escribe nada en este notebook. Las variables anteriores son del entorno, no del schema.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.

Comprobamos que `INFLUXDB_URL` está cargado y que `python-dotenv` está disponible (si no, usamos el parser propio).


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Inspeccionamos las variables clave (sin imprimir el token completo).


In [ ]:
import os

def masked(value: str | None) -> str:
    if not value:
        return "<vacío>"
    return value[:4] + "*" * (max(0, len(value) - 4))

print("INFLUXDB_URL:", os.environ.get("INFLUXDB_URL", "<no definido>"))
print("INFLUXDB_ORG:", os.environ.get("INFLUXDB_ORG", "<no definido>"))
print("INFLUXDB_TOKEN:", masked(os.environ.get("INFLUXDB_TOKEN")))
print("OFFLINE MODE:", os.environ.get("INFLUX_OFFLINE", "false"))


## 10. Exploración paso a paso

Intentamos abrir un cliente; si no se puede (sin servicios o sin `influxdb_client` instalado), mostramos cómo continuaríamos en modo offline.


In [ ]:
client = get_influx_client(allow_offline=True)
if client is None:
    print("Modo offline: trabajaremos con mocks de notebooks/_common/synthetic_mocks.py")
else:
    health = client.health()
    print("Conectado:", health.status, "—", health.message)


## 11. Transformación bronce → plata

No aplica — este notebook prepara la herramienta, no la usa.


## 12. Construcción de capa oro

No aplica.


## 13. Visualizaciones explicativas

Pintamos una pequeña gráfica de la disponibilidad de servicios para documentar visualmente nuestro estado de stack en clase.


In [ ]:
import os

def _is_truthy(env_var: str) -> bool:
    return os.environ.get(env_var, "").lower() in {"1", "true", "yes"}

estado = pd.Series(
    {
        "InfluxDB cliente": "ok" if client is not None else "offline",
        "python-dotenv": "instalado" if "dotenv" in dir() or os.environ.get("INFLUXDB_URL") else "fallback",
        "Modo OFFLINE explícito": "sí" if _is_truthy("INFLUX_OFFLINE") else "no",
    },
    name="estado",
)
estado.to_frame()


## 14. Validaciones

Comprobaciones mínimas que un alumno debería pasar antes de continuar.


In [ ]:
url = os.environ.get("INFLUXDB_URL")
token = os.environ.get("INFLUXDB_TOKEN")
assert url, "INFLUXDB_URL no definido — copiar .env.example a .env"
assert token, "INFLUXDB_TOKEN no definido — pedir credenciales a profesor"
assert "CHANGE_ME" not in token, "Sigue el placeholder de .env.example: regenerar con openssl"
print("Validaciones OK")


## 15. Errores comunes

1. **`INFLUXDB_TOKEN=CHANGE_ME...`** sin reemplazar. Generar con `openssl rand -hex 32`.
2. **Olvidar arrancar el stack** (`make demo` o `task up`).
3. **Usar `localhost:8086` en lugar de `8087`** que es el puerto host.
4. **No exportar las variables al kernel** — Jupyter Lab no recarga `.env` automáticamente al editarlo; reiniciar kernel.
5. **Commitear `.env`**. Está en `.gitignore` por algo.


## 16. Ejercicios propuestos

1. Añade un decorador `@requires_influx` que redirija a una función fallback con mocks si `client is None`.
2. Implementa una función `ping(url)` que compruebe `/health` con `httpx`.
3. Escribe la función `secret_okay(token)` que valide longitud mínima y ausencia de placeholders.


## 17. Cómo se reutiliza con datos reales

Para conectar a `simarro-prod` (producción CAPTIA): cambiar `INFLUXDB_URL` al endpoint LAN o Tailscale del IES, usar `edu-token-simarro` con permiso read-only, y dejar `INFLUX_OFFLINE` sin definir. El resto del código no cambia.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `01_case_A_pipeline_iot/01_explicacion_pipeline_centinela.ipynb`.
- Documento web del caso: `docs/operations/environment.md`.
- La política de secretos del repo está en `docs/operations/environment.md`.
